# 02 — Data Preparation

## Project: Sales Anomaly Detection for a Multi-Outlet F&B Business

### Objective

The goal of this notebook is to prepare clean and structured datasets for anomaly detection.

In the previous notebook, the raw data was explored and key findings were identified, including:

- Each row represents one receipt-level transaction.
- The dataset contains 53,820 transactions.


In [1]:
import pandas as pd
import numpy as np

# Load data
transactions = pd.read_csv("../data/raw/Transactions.csv")
products = pd.read_csv("../data/raw/Products.csv")

# Check output
print(transactions.shape)
print(products.shape)

transactions.head()

(53820, 11)
(96, 6)


,Outlet,Date,Time,NetSales,Tax,TotalAmount,ReceiptNumber,Items,TotalItem,PaymentMethod,UseLoyaltyCard
0,SHOP001,2025-01-02,07:01:05,20720.721,2279.279,23000,SHOP001JAN2500001,Basic Latte (Ice Arabica),1,BCA,0
1,SHOP001,2025-01-02,07:04:11,18018.018,1981.982,20000,SHOP001JAN2500002,BLACK (Houseblend),1,BCA,0
2,SHOP002,2025-01-02,07:10:46,39639.640,4360.360,44000,SHOP002JAN2500001,"Friendly Coffee (Hot), Shakencano",2,Cash,0
3,SHOP002,2025-01-02,07:14:25,36036.036,3963.964,40000,SHOP002JAN2500002,"BLACK (Arabica), Americano (Hot Arabica)",2,Cash,0
4,SHOP002,2025-01-02,07:23:05,36036.036,3963.964,40000,SHOP002JAN2500003,"WHITE (Arabica), Susu Oat",2,Cash,0


In [2]:
transactions.columns = (
    transactions.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

products.columns = (
    products.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

transactions = transactions.rename(columns={
    "netsales": "net_sales",
    "totalamount": "total_amount",
    "receiptnumber": "receipt_number",
    "totalitem": "total_item",
    "paymentmethod": "payment_method",
    "useloyaltycard": "use_loyalty_card"
})

products = products.rename(columns={
    "productid": "product_id",
    "productname": "product_name"
})

transactions.head()

,outlet,date,time,net_sales,tax,total_amount,receipt_number,items,total_item,payment_method,use_loyalty_card
0,SHOP001,2025-01-02,07:01:05,20720.721,2279.279,23000,SHOP001JAN2500001,Basic Latte (Ice Arabica),1,BCA,0
1,SHOP001,2025-01-02,07:04:11,18018.018,1981.982,20000,SHOP001JAN2500002,BLACK (Houseblend),1,BCA,0
2,SHOP002,2025-01-02,07:10:46,39639.640,4360.360,44000,SHOP002JAN2500001,"Friendly Coffee (Hot), Shakencano",2,Cash,0
3,SHOP002,2025-01-02,07:14:25,36036.036,3963.964,40000,SHOP002JAN2500002,"BLACK (Arabica), Americano (Hot Arabica)",2,Cash,0
4,SHOP002,2025-01-02,07:23:05,36036.036,3963.964,40000,SHOP002JAN2500003,"WHITE (Arabica), Susu Oat",2,Cash,0


In [3]:
transactions["date"] = pd.to_datetime(transactions["date"])

transactions["transaction_datetime"] = pd.to_datetime(
    transactions["date"].astype(str) + " " + transactions["time"]
)

transactions["hour"] = transactions["transaction_datetime"].dt.hour


In [4]:
transactions.shape
transactions.columns
transactions.head()

,outlet,date,time,net_sales,tax,total_amount,receipt_number,items,total_item,payment_method,use_loyalty_card,transaction_datetime,hour
0,SHOP001,2025-01-02,07:01:05,20720.721,2279.279,23000,SHOP001JAN2500001,Basic Latte (Ice Arabica),1,BCA,0,2025-01-02 07:01:05,7
1,SHOP001,2025-01-02,07:04:11,18018.018,1981.982,20000,SHOP001JAN2500002,BLACK (Houseblend),1,BCA,0,2025-01-02 07:04:11,7
2,SHOP002,2025-01-02,07:10:46,39639.640,4360.360,44000,SHOP002JAN2500001,"Friendly Coffee (Hot), Shakencano",2,Cash,0,2025-01-02 07:10:46,7
3,SHOP002,2025-01-02,07:14:25,36036.036,3963.964,40000,SHOP002JAN2500002,"BLACK (Arabica), Americano (Hot Arabica)",2,Cash,0,2025-01-02 07:14:25,7
4,SHOP002,2025-01-02,07:23:05,36036.036,3963.964,40000,SHOP002JAN2500003,"WHITE (Arabica), Susu Oat",2,Cash,0,2025-01-02 07:23:05,7


## Zero-Value Transaction Investigation

### What we are doing

In this step, we investigate transactions where the final transaction amount is zero.

The previous data understanding phase showed that there are 1,733 transactions with:

- `net_sales = 0`
- `total_amount = 0`

We also observed that the number of loyalty card transactions is exactly 1,733.

### Why we are doing this

Zero-value transactions may represent valid business behavior, such as loyalty rewards or free redemptions.

However, they may also represent cancelled transactions, voided receipts, test records, or data quality issues.

Before preparing the modeling dataset, we need to understand whether these rows should be:

- kept,
- removed,
- flagged,
- or analyzed separately.

### Expected outcome

By the end of this step, we should understand the relationship between zero-value transactions and loyalty card usage.

This will help us decide how to treat zero-value transactions during data preparation.

In [6]:
# Create a separate dataframe for zero-value transactions
zero_sales_tx = transactions[
    transactions["total_amount"] == 0
].copy()

# Check number of zero-value transactions
print("Zero-value transactions:", len(zero_sales_tx))

# Preview zero-value transactions
zero_sales_tx.head()

Zero-value transactions: 1733


,outlet,date,time,net_sales,tax,total_amount,receipt_number,items,total_item,payment_method,use_loyalty_card,transaction_datetime,hour
31,SHOP001,2025-01-02,10:37:45,0.0,0.0,0,SHOP001JAN2500013,Cappuccino (Hot Houseblend),1,Cash,1,2025-01-02 10:37:45,10
37,SHOP002,2025-01-02,10:59:44,0.0,0.0,0,SHOP002JAN2500016,Friendly Coffee (Gedhe),1,Cash,1,2025-01-02 10:59:44,10
78,SHOP001,2025-01-02,13:58:16,0.0,0.0,0,SHOP001JAN2500038,Cappuccino (Ice Houseblend),1,Cash,1,2025-01-02 13:58:16,13
85,SHOP002,2025-01-02,14:35:33,0.0,0.0,0,SHOP002JAN2500025,Not Tiramisu Latte (Gedhe),1,Cash,1,2025-01-02 14:35:33,14
86,SHOP003,2025-01-02,14:38:03,0.0,0.0,0,SHOP003JAN2500019,Waffle,1,Cash,1,2025-01-02 14:38:03,14


In [7]:
transactions["is_zero_value_transaction"] = transactions["total_amount"] == 0

In [8]:
pd.crosstab(
    transactions["use_loyalty_card"],
    transactions["is_zero_value_transaction"]
)

is_zero_value_transaction,False,True
use_loyalty_card,,
0,52087,0
1,0,1733


### Finding — Zero-Value Transactions and Loyalty Card Usage

A cross-tabulation was created between `use_loyalty_card` and `is_zero_value_transaction`.

The result showed a perfect relationship:

- All transactions where `use_loyalty_card = 0` are non-zero transactions.
- All transactions where `use_loyalty_card = 1` are zero-value transactions.

This means that all 1,733 zero-value transactions are linked to loyalty card usage.

### Interpretation

The zero-value transactions are unlikely to be random data errors.

They likely represent a specific business process, such as:

- loyalty reward redemption,
- free item redemption,
- promotional loyalty transactions,
- or another POS-defined loyalty behavior.

### Data preparation decision

Zero-value transactions will not be removed immediately.

Instead, they will be flagged using the column:

`is_zero_value_transaction`

This allows future analysis to treat these transactions separately.

### Implications for anomaly detection

Zero-value transactions can distort revenue-based anomaly detection because they reduce transaction amount and daily revenue.

However, they may still be valid business events.

Therefore, future modeling may use separate datasets:

- a full transaction dataset,
- a paid transaction dataset excluding zero-value transactions,
- and a zero-value transaction subset for separate investigation.

In [9]:
zero_sales_tx["outlet"].value_counts()

outlet
SHOP001    765
SHOP003    562
SHOP002    406
Name: count, dtype: int64

In [10]:
zero_sales_tx["payment_method"].value_counts()

payment_method
Cash    1733
Name: count, dtype: int64

In [11]:
zero_sales_tx["hour"].value_counts().sort_index()

hour
7      70
8      79
9      90
10    115
11     97
12    114
13     96
14     85
15    105
16    112
17    140
18    118
19    127
20    121
21    142
22    122
Name: count, dtype: int64